# MitoFusionAI Demo Notebook

Reproduces the MitoFusionAI pipeline on a 1,000-compound stratified sample drawn from PubChem BioAssay AID 1362.

**Inputs (provided in this repository):**
- `sample_training_set_1000.csv` — 1,000 compounds stratified to preserve the 2.07% active-class prevalence

**Pipeline:**
1. Load + verify SMILES
2. Featurize (91 descriptors: 85 RDKit fragments + 6 physicochemical)
3. Scaffold-stratified 80/20 split
4. Train XGBoost with the manuscript's optimized hyperparameters
5. Evaluate (ROC-AUC, PR-AUC, MCC, balanced accuracy)
6. SHAP feature importance (top 15)

**Reference:** Senary AM, Soliman AM. Machine learning prediction of mitochondrial disruptors...

In [ ]:
import numpy as np, pandas as pd
from rdkit import Chem
from rdkit.Chem import Descriptors, Fragments
from rdkit.Chem.Scaffolds import MurckoScaffold
import xgboost as xgb
from sklearn.metrics import roc_auc_score, average_precision_score, matthews_corrcoef, balanced_accuracy_score
np.random.seed(42)
print('Libraries loaded.')
# Install if missing:  pip install rdkit xgboost scikit-learn pandas numpy shap

In [ ]:
df = pd.read_csv('sample_training_set_1000.csv')
print(f'Loaded {len(df)} compounds ({df.active_label.sum()} actives, {100*df.active_label.mean():.1f}% prevalence)')
df.head()

In [ ]:
FRAGS = [(n, getattr(Fragments, n)) for n in sorted(n for n in dir(Fragments) if n.startswith('fr_'))]
print(f'Using {len(FRAGS)} fragment descriptors + 6 physicochemical = 91 features')

def featurize(smi):
    m = Chem.MolFromSmiles(smi)
    if m is None: return [0.0]*91
    return ([f(m) for _, f in FRAGS] +
            [Descriptors.MolWt(m), Descriptors.MolLogP(m), Descriptors.TPSA(m),
             Descriptors.NumRotatableBonds(m), Descriptors.NumHDonors(m), Descriptors.NumHAcceptors(m)])

X = np.array([featurize(s) for s in df.canonical_smiles], dtype=np.float32)
y = df.active_label.values
print(f'Feature matrix: {X.shape}')

In [ ]:
# Scaffold-stratified 80/20 split
df['scaffold'] = [MurckoScaffold.MurckoScaffoldSmiles(mol=Chem.MolFromSmiles(s), includeChirality=False) if Chem.MolFromSmiles(s) else '' for s in df.canonical_smiles]
scaffs = list(df.groupby('scaffold').groups.items())
scaffs.sort(key=lambda kv: -len(kv[1]))
train_idx, test_idx = [], []
for i, (s, idx) in enumerate(scaffs):
    (test_idx if i % 5 == 0 else train_idx).extend(idx)
X_train, X_test = X[train_idx], X[test_idx]
y_train, y_test = y[train_idx], y[test_idx]
print(f'Train: {len(train_idx)} ({y_train.sum()} actives)  Test: {len(test_idx)} ({y_test.sum()} actives)')

In [ ]:
# Train XGBoost with the manuscript's optimized hyperparameters
model = xgb.XGBClassifier(
    n_estimators=400, max_depth=9, learning_rate=0.0635,
    scale_pos_weight=41.59, reg_alpha=0.199, reg_lambda=0.0043,
    subsample=0.994, colsample_bytree=0.646,
    max_delta_step=1, min_child_weight=10,
    objective='binary:logistic', tree_method='hist', random_state=42,
)
model.fit(X_train, y_train)
p = model.predict_proba(X_test)[:, 1]

print(f'ROC-AUC:           {roc_auc_score(y_test, p):.4f}')
print(f'PR-AUC:            {average_precision_score(y_test, p):.4f}')
print(f'MCC:               {matthews_corrcoef(y_test, p >= 0.5):.4f}')
print(f'Balanced accuracy: {balanced_accuracy_score(y_test, p >= 0.5):.4f}')
print()
print('Note: small sample produces wider metric variance than the full 194,235-compound dataset.')
print('Full-dataset results (manuscript): ROC-AUC 0.849, PR-AUC 0.157, MCC 0.191, BA 0.69')

In [ ]:
# SHAP feature importance (top 15)
import shap
explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_test)
feat_names = ([n for n, _ in FRAGS] + ['MolWt', 'MolLogP', 'TPSA', 'NumRotatableBonds', 'NumHDonors', 'NumHAcceptors'])
shap.summary_plot(shap_values, X_test, feature_names=feat_names, max_display=15, show=True)